# Exploring Food Compound Taste with the FooDB Database

---

In this exercise, we will practice applying a **machine learning model** to real-world chemical data and interpreting the results visually.

You will use the **FooDB compound database** and the **bitter/sweet classifier** trained in the previous session.
Remember to **fill in the missing code** in each exercise cell.

---

| Step | Task | Skill practised |
|------|------|-----------------|
| 1 | Import libraries | |
| 2 | Set up configuration |  |
| 3 | Load & inspect the dataset | pandas, `ast.literal_eval` |
| 4 | Count molecules per food source | `explode`, `value_counts`, bar charts |
| 5 | Load the trained model | `pickle`, model introspection |
| 6 | Convert SMILES → fingerprints | RDKit, looping over data |
| 7 | Run predictions | Applying `model.predict()` |
| 8 | Run PCA | `sklearn.decomposition.PCA` |
| 9 | Plot PCA by superclass | Interpreting dimensionality reduction |
| 10 | Plot PCA by food source | Parsing string-lists, grouping, multi-panel plots |

---

### How to use this notebook

- Cells marked ** Exercise** contain `# YOUR CODE HERE` placeholders — these are the ones you fill in.
- Cells marked ** Hint** have the solution.
- Every other cell is ready to run as-is.

---

> **Required libraries:** `rdkit`, `scikit-learn`, `pandas`, `numpy`, `matplotlib`  
> Install with: `pip install rdkit scikit-learn pandas numpy matplotlib`

> ⚠️ **Workshop participants:** You are viewing a read-only copy from GitHub.
> To save your work, go to **File → Save a copy in Drive**.

## Clone the repository to have all the necessary files

In [ ]:
import os

repo_url = "https://github.com/DiPizio-Lab/4thFLAVOURsomeWS_AI4FlavourInnovation.git"
repo_name = "4thFLAVOURsomeWS_AI4FlavourInnovation"

if not os.path.exists(repo_name):
    !git clone {repo_url}

os.chdir(repo_name)

## Some useful links:

| Library | Link |
|---------|------|
| **NumPy** | https://numpy.org/doc/stable/user/quickstart.html |
| **Pandas** | https://pandas.pydata.org/docs/getting_started/intro_tutorials/ |
| **RDKit** | https://www.rdkit.org/docs/GettingStartedInPython.html |
| **Scikit-learn PCA** | https://scikit-learn.org/stable/modules/decomposition.html#pca |
| **Matplotlib** | https://matplotlib.org/stable/users/explain/quick_start.html |

In [ ]:
!pip install rdkit scikit-learn pandas numpy matplotlib

---
## Step 1 — Import the Libraries We Need

We need several libraries for this notebook. Here is what each one does:

| Library / Module | What it does |
|---|---|
| `pandas` | Loads and manipulates tables of data |
| `numpy` | Fast numerical arrays |
| `matplotlib` | Creates charts and plots |
| `ast` | Parses text strings back into Python objects (e.g. `"['apple', 'grape']"` → `['apple', 'grape']`) |
| `pickle` | Loads our saved machine learning model from disk |
| `rdkit` | Reads chemical structures from SMILES strings |
| `sklearn.decomposition.PCA` | Reduces 1024-bit fingerprints to 2D for plotting |

**Why `ast`?**  
When you save a Python list to a CSV file, it becomes the *text* `"['apple', 'grape']"` — a string that *looks* like a list.  
`ast.literal_eval` safely converts that string back into a real Python list.


In [ ]:
# --- Standard Python libraries ---
import ast                        # convert stored list-strings back to real Python lists
import pickle                     # load the saved model from disk
import numpy as np                # arrays and math
import pandas as pd               # tables (DataFrames)

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches



# --- Chemistry library: RDKit ---
from rdkit import Chem                                            # parse molecules from SMILES
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator  # compute Morgan fingerprints

# --- Machine learning: scikit-learn ---
from sklearn.decomposition import PCA

print("All imports successful!")

---
## Step 2 — Configuration

It is good practice to collect all file paths and settings at the top of a notebook in one place.  
That way, if you rename a file or change a setting, **you only need to edit one cell**.

> ⚠️ Make sure `MODEL_PATH` points to the `.pkl` file saved by the training notebook.  
> The default `'bittersweet_model_best_fold.pkl'` matches the name used there.

This cell is already complete — just run it and read the output.

In [ ]:
# --- File paths ---
FOODB_PATH  = 'data' + os.sep + 'FooDB.csv'       # the FooDB database
MODEL_PATH  = 'bittersweet_model_best_fold.pkl'   # the trained Random Forest

# --- Morgan fingerprint settings ---
# These MUST match the settings used when the model was trained!
MORGAN_RADIUS = 2     # how far around each atom to look
MORGAN_NBITS  = 1024  # length of the fingerprint bit vector

# --- Plot settings ---
TOP_N_FOODS = 9   # how many top food sources to visualise (3x3 grid)

print("Configuration set.")
print(f"  FooDB file  : {FOODB_PATH}")
print(f"  Model file  : {MODEL_PATH}")
print(f"  Fingerprint : radius={MORGAN_RADIUS}, bits={MORGAN_NBITS}")
print(f"  Top N foods : {TOP_N_FOODS}")

---
## Step 3 — Load the FooDB Dataset

We will load the CSV using pandas.  
Two important parameters:
- `header=0` — the **first row** is the column names.
- `index_col=0` — the **first column** is the row index (here: the InChIKey, a unique molecular identifier).

After loading, the `food_sources` column will look like a text string:  
`"['apple', 'grape', 'orange']"`  

We need to convert it back to an actual Python list using `ast.literal_eval`.

---
### ✏️ Exercise 3.1 — Load the CSV

In [ ]:
# YOUR CODE HERE
# Load FOODB_PATH using pd.read_csv
# Remember: header=0, index_col=0
# Store the result in a variable called df_foodb


print(f"Loaded {len(df_foodb)} rows and {len(df_foodb.columns)} columns.")
print(f"Column names: {df_foodb.columns.tolist()}")
print()
df_foodb.head()

<details>
<summary>💡 Hint</summary>

```python
df_foodb = pd.read_csv(FOODB_PATH, header=0, index_col=0)
```
</details>

### ✏️ Exercise 3.2 — Convert `food_sources` from string to list

The `food_sources` column currently contains strings like `"['apple', 'grape']"`.  
We need to convert each one to a real Python list.

- Use `df['column'].apply(...)` to apply a function to every row.
- Use `ast.literal_eval(x)` to parse the string.
- If the value is not a string (e.g. it is `NaN`), return an empty list `[]`.
- Use `isinstance(x, str)` to check if `x` is a string.

In [ ]:
# YOUR CODE HERE
# Apply ast.literal_eval to convert each entry in df_foodb['food_sources']
# If the value is not a string, return []
# Assign the result back to df_foodb['food_sources']


# Quick check: show the first 3 entries
print("First 3 food_sources entries:")
for i, sources in enumerate(df_foodb['food_sources'].head(3)):
    print(f"  Row {i}: {sources}")

<details>
<summary>💡 Hint</summary>

Use a lambda inside `.apply()`:

```python
df_foodb['food_sources'] = df_foodb['food_sources'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else []
)
```
</details>

---
## Step 4 — Count Molecules per Food Source

Each compound can appear in **multiple food sources** (e.g. caffeine is found in coffee, tea, and chocolate).  
The `food_sources` column contains a list per row.

To count compounds per food, we need two pandas tricks:
1. **`.explode()`** — transforms each element of a list into a separate row.  
   e.g. one row with `['apple', 'grape']` → two rows: `'apple'` and `'grape'`.
2. **`.value_counts()`** — counts how many times each value appears.

---
### ✏️ Exercise 4.1 — Count compounds per food source

In [ ]:
# YOUR CODE HERE
# 1. Take the 'food_sources' column of df_foodb
# 2. Call .explode() on it to flatten the lists
# 3. Call .value_counts() to count each food source
# 4. Store the result in a variable called final_counts


print(f"Number of unique food sources found: {len(final_counts)}")
print()
print("Top 20 most common food sources:")
print(final_counts.head(20).to_string())

<details>
<summary>💡 Hint</summary>

```python
final_counts = df_foodb['food_sources'].explode().value_counts()
```
</details>

### ✏️ Exercise 4.2 — Plot the top 20 food sources as a bar chart

Now create a bar chart of the 20 most common food sources.  
The full plotting code is provided — your job is to fill in the two blank lines.

**Think about:**
- What should go on the x-axis? (hint: food names)
- What should go on the y-axis? (hint: compound counts)

In [ ]:
top20 = final_counts.head(20)

fig, ax = plt.subplots(figsize=(12, 5))

# YOUR CODE HERE — draw the bars
# ax.bar(x_positions, heights, ...)
# x_positions: range(len(top20))
# heights: the compound counts
# Use color='steelblue', edgecolor='white'


ax.set_xticks(range(len(top20)))

# YOUR CODE HERE — set the x tick labels
# Use top20.index for the labels, rotation=45, ha='right', fontsize=9


ax.set_ylabel('Number of compounds')
ax.set_title('Top 20 Food Sources by Number of Compounds in FooDB')
plt.tight_layout()
plt.show()

# Save the top-N food source names for later
top_foods = final_counts.head(TOP_N_FOODS).index.tolist()
print(f"\nTop {TOP_N_FOODS} food sources we will plot:")
for i, food in enumerate(top_foods, 1):
    print(f"  {i}. {food}  ({final_counts[food]} compounds)")

<details>
<summary>💡 Hint</summary>

```python
ax.bar(range(len(top20)), top20.values, color='steelblue', edgecolor='white')
ax.set_xticklabels(top20.index, rotation=45, ha='right', fontsize=9)
```
</details>

---
## Step 5 — Load the Trained Bitter/Sweet Model

In the companion notebook, we trained a **Random Forest classifier** and saved it to a `.pkl` file using `pickle`.

**What is pickle?**  
`pickle` is Python's built-in way to **serialise** (save) and **deserialise** (load) almost any Python object — including scikit-learn models.  
Once loaded, you can call `.predict()` on it exactly as if you had just trained it.

**Key syntax:**
```python
with open('file.pkl', 'rb') as f:   # 'rb' = read binary
    obj = pickle.load(f)
```

---
### ✏️ Exercise 5.1 — Load the model and inspect it

In [ ]:
# YOUR CODE HERE
# Open MODEL_PATH in binary-read mode and use pickle.load() to load the model
# Store it in a variable called 'model'


print(f"Model loaded from: {MODEL_PATH}")
print(f"Model type       : {type(model).__name__}")
print(f"Number of trees  : {model.n_estimators}")
print(f"Expected features: {model.n_features_in_} (should match MORGAN_NBITS = {MORGAN_NBITS})")

<details>
<summary>💡 Hint</summary>

```python
with open(MODEL_PATH, 'rb') as f:
    model = pickle.load(f)
```
</details>

**Think about it:**
After running the cell, look at `model.n_features_in_`. Why must this match `MORGAN_NBITS`?

---
## Step 6 — Convert SMILES to Morgan Fingerprints

Machine learning models only understand **numbers**, not chemical text.  
We need to convert each molecule's SMILES string into a **Morgan fingerprint** — a fixed-length binary vector (1024 zeros and ones) that encodes the chemical environment around each atom.

**Key concepts:**
- **SMILES** — a text description of a molecule (e.g. `CCO` = ethanol, `CN1C=NC2=C1C(=O)N(C(=O)N2C)C` = caffeine)
- **Morgan fingerprint** — a numeric representation we can feed to a model
- Some SMILES are invalid — we skip those and store `NaN` for their predictions

The helper function `smiles_to_morgan` is already written for you.  
Your job is to fill in the loop that uses it.

---
### ✏️ Exercise 6.1 — Complete the fingerprint generation loop

In [ ]:
# Set up the Morgan fingerprint generator
generator = GetMorganGenerator(radius=MORGAN_RADIUS, fpSize=MORGAN_NBITS)

def smiles_to_morgan(smiles):
    """Convert a SMILES string to a Morgan fingerprint (numpy array).
    Returns None if the SMILES is invalid or cannot be parsed.
    """
    if not isinstance(smiles, str) or smiles.strip() == '':
        return None
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return generator.GetFingerprintAsNumPy(mol)


print("Computing Morgan fingerprints for all compounds...")
print("(This may take few seconds for ~10,000 molecules)")
print()

fingerprints = []   # will store one numpy array per molecule
valid_mask   = []   # True if fingerprint was computed, False if SMILES was invalid

# YOUR CODE HERE
# Loop over each SMILES string in df_foodb['SMILES']
# For each one:
#   - Call smiles_to_morgan(smiles) to get a fingerprint
#   - If the fingerprint is not None:
#       * append the fingerprint to 'fingerprints'
#       * append True to 'valid_mask'
#   - Else (invalid SMILES):
#       * append np.zeros(MORGAN_NBITS, dtype=np.uint8) to 'fingerprints'  (placeholder)
#       * append False to 'valid_mask'


valid_mask = np.array(valid_mask)
X_all      = np.vstack(fingerprints)   # shape: (n_molecules, 1024)

print(f"Total molecules  : {len(valid_mask)}")
print(f"Valid SMILES     : {valid_mask.sum()}")
print(f"Invalid / skipped: {(~valid_mask).sum()}")
print(f"Fingerprint array shape: {X_all.shape}")

<details>
<summary>💡 Hint</summary>

```python
for smiles in df_foodb['SMILES']:
    fp = smiles_to_morgan(smiles)
    if fp is not None:
        fingerprints.append(fp)
        valid_mask.append(True)
    else:
        fingerprints.append(np.zeros(MORGAN_NBITS, dtype=np.uint8))
        valid_mask.append(False)
```
</details>

---
## Step 7 — Run Predictions

Now we pass the fingerprints through the loaded model to get a **prediction** for each compound.

The model outputs:
- `1` → **bitter**
- `0` → **sweet**

We only predict on rows where the SMILES was valid (`valid_mask == True`).  
For invalid rows, we leave the prediction as `NaN`.

---
### ✏️ Exercise 7.1 — Apply the model and add a `taste_label` column

Fill in the two blanks:
1. Call `model.predict()` on the valid fingerprints only.
2. Use `.map()` to convert `1.0` → `'bitter'` and `0.0` → `'sweet'`.

In [ ]:
print("Running predictions...")

# Start with NaN for every row
predictions = np.full(len(df_foodb), np.nan)

# YOUR CODE HERE
# Fill in predictions for valid rows using model.predict()
# Hint: use valid_mask to index into both X_all and predictions


# Add predictions to the DataFrame
df_foodb['prediction'] = predictions

# YOUR CODE HERE
# Add a 'taste_label' column that maps 1.0 -> 'bitter' and 0.0 -> 'sweet'
# Hint: use .map() with a dictionary


n_bitter = (df_foodb['taste_label'] == 'bitter').sum()
n_sweet  = (df_foodb['taste_label'] == 'sweet').sum()
n_na     = df_foodb['taste_label'].isna().sum()

print(f"Prediction summary:")
print(f"  Predicted bitter : {n_bitter:>7,}")
print(f"  Predicted sweet  : {n_sweet:>7,}")
print(f"  Could not predict: {n_na:>7,}  (invalid SMILES)")
print()
df_foodb[['SMILES', 'food_sources', 'class', 'superclass', 'taste_label']].head()

<details>
<summary>💡 Hint</summary>

```python
predictions[valid_mask] = model.predict(X_all[valid_mask])

df_foodb['taste_label'] = df_foodb['prediction'].map({1.0: 'bitter', 0.0: 'sweet'})
```
</details>

**Think about it:**
Look at the proportion of bitter vs sweet predictions. Does the imbalance surprise you?  
What might cause it? (Hint: think about the training data.)

---
## Step 8 — Dimensionality Reduction with PCA

Each molecule is represented as a **1024-dimensional fingerprint**.  
Humans cannot visualise 1024 dimensions — so we use **PCA** to compress each vector to just **2 numbers** while retaining as much structure as possible.

**How PCA works (simplified):**
1. It finds the directions in the 1024-dimensional space along which the data varies the most.
2. It projects every point onto the two most important directions (PC1 and PC2).
3. The result is a 2D coordinate for every molecule that we can plot.

Molecules **close together** in the PCA plot tend to have **similar chemical structures**.

> We fit PCA on **all valid fingerprints** so the coordinate system is global and consistent across all our plots.

---
### ✏️ Exercise 8.1 — Fit PCA and transform the fingerprints

In [ ]:
print("Fitting PCA on all valid fingerprints...")

X_valid = X_all[valid_mask]   # shape: (n_valid, 1024)

# YOUR CODE HERE
# 1. Create a PCA object with n_components=2 and random_state=42
# 2. Fit it on X_valid
# 3. Transform X_valid to get 2D coordinates
#    Store the result in X_pca_valid (shape: (n_valid, 2))


# Fill results back into a full array (NaN for invalid rows)
X_pca_all = np.full((len(df_foodb), 2), np.nan)
X_pca_all[valid_mask] = X_pca_valid   # use the variable you created above

# Add PC1 and PC2 as columns
df_foodb['PC1'] = X_pca_all[:, 0]
df_foodb['PC2'] = X_pca_all[:, 1]

var1 = pca.explained_variance_ratio_[0] * 100
var2 = pca.explained_variance_ratio_[1] * 100

print(f"PCA fitted.")
print(f"  PC1 explains {var1:.1f}% of variance")
print(f"  PC2 explains {var2:.1f}% of variance")
print(f"  Together     {var1 + var2:.1f}% of variance captured in 2D")

<details>
<summary>💡 Hint</summary>

```python
pca = PCA(n_components=2, random_state=42)
pca.fit(X_valid)
X_pca_valid = pca.transform(X_valid)
```

Or more concisely with `fit_transform`:
```python
pca = PCA(n_components=2, random_state=42)
X_pca_valid = pca.fit_transform(X_valid)
```
</details>

**Think about it:**
PC1 and PC2 together explain only a small fraction of the total variance.  
What does that mean about how much information we lose by going from 1024D to 2D?

---
## Step 9 — Plot PCA Coloured by Predicted Taste (per Superclass)

We now create a **3×3 grid of scatter plots**, one panel per compound superclass.

For each superclass:
1. Select only molecules annotated with that superclass.
2. Plot them in the global PCA coordinate system.
3. Colour each dot by predicted taste: **blue = bitter**, **orange = sweet**.

**Reading the plots:**
- Each **dot** is one compound.
- Dots close together = similar molecular structures.
- The x/y axes are abstract directions in chemical space — no simple chemical meaning, but they preserve distances.

This cell is complete — run it and study the output.

In [ ]:
TASTE_SUPERCLASSES = [
    "Organooxygen compounds",
    "Organic acids and derivatives",
    "Benzenoids",
    "Phenylpropanoids and polyketides",
    "Lipids and lipid-like molecules",
    "Alkaloids and derivatives",
    "Organoheterocyclic compounds",
    "Organosulfur compounds",
    "Organonitrogen compounds"
]

TASTE_COLORS = {'bitter': 'steelblue', 'sweet': 'coral'}

df_valid = df_foodb[
    df_foodb["superclass"].notna() &
    df_foodb["taste_label"].notna()
]
df_valid = df_valid[df_valid["superclass"].isin(TASTE_SUPERCLASSES)]

fig, axes = plt.subplots(3, 3, figsize=(15, 14))
axes = axes.flatten()

for ax_idx, sc in enumerate(TASTE_SUPERCLASSES):
    ax = axes[ax_idx]
    subset = df_valid[df_valid["superclass"] == sc]

    for taste, color in TASTE_COLORS.items():
        group = subset[subset["taste_label"] == taste]
        ax.scatter(
            group["PC1"], group["PC2"],
            c=color, alpha=0.45, s=12, linewidths=0, label=taste
        )

    n_total  = len(subset)
    n_bitter = (subset["taste_label"] == "bitter").sum()
    n_sweet  = (subset["taste_label"] == "sweet").sum()

    ax.set_title(
        f"{sc}\n(n={n_total} | bitter={n_bitter} sweet={n_sweet})",
        fontsize=9, fontweight="bold"
    )
    ax.set_xlabel(f"PC1 ({var1:.1f}% var)", fontsize=7)
    ax.set_ylabel(f"PC2 ({var2:.1f}% var)", fontsize=7)
    ax.tick_params(labelsize=6)

legend_handles = [
    mpatches.Patch(color=TASTE_COLORS['bitter'], label='Bitter (predicted)'),
    mpatches.Patch(color=TASTE_COLORS['sweet'],  label='Sweet (predicted)'),
]
fig.legend(handles=legend_handles, loc='lower center', ncol=2,
           fontsize=11, frameon=True, bbox_to_anchor=(0.5, -0.03))
fig.suptitle(
    "PCA of Morgan Fingerprints — coloured by predicted taste\n(grouped by compound superclass)",
    fontsize=13, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()

| **Superclass**                     | **Typical Taste** | **Notes**                                                               |
|------------------------------------|-------------------|-------------------------------------------------------------------------|
| **Phenylpropanoids & polyketides** | Mostly bitter     | Sweet glycosides (e.g., stevioside) but many bitter flavonoids/tannins. |
| **Benzenoids**                     | Sweet             | Many aromatic sweet-smelling compounds.                                 |
| **Lipids & lipid-like molecules**  | Neutral → Bitter  | Fatty/neutral; oxidized lipids and steroids often bitter.               |
| **Organooxygen compounds**         | Sweet             | Includes sugars, sugar alcohols, esters, and glycosides.                |
| **Organic acids & derivatives**    | Sweet / Sour      | Lactones and esters often sweet; free acids are sour.                   |
| **Alkaloids & derivatives**        | Strongly Bitter   | Caffeine, quinine, nicotine, and similar compounds.                     |
| **Organoheterocyclic compounds**   | Bitter            | Many plant defense molecules; frequently activate bitter receptors.     |
| **Organosulfur compounds**         | Bitter / Pungent  | Garlic/onion-type compounds; often sharp or bitter.                     |
| **Organonitrogen compounds**       | Bitter            | Nitrogen-containing compounds commonly activate bitter receptors.       |

---
## Step 10 — PCA Coloured by Predicted Taste (per Food Source)

In Step 9, we split the PCA by **compound superclass** (a chemical classification).  
Now let's answer a different question:

> **"Do compounds from the same food tend to cluster together in chemical space, and how bitter/sweet is each food?"**

This is trickier than the superclass plot because:
- The `food_sources` column contains **lists**, not single values — one molecule can belong to many foods.
- We need to **explode** the list column, creating one row per (molecule, food) pair, before we can group by food.

---

### 🔑 Key concept: exploding a list column with additional columns

In Step 4 we called `.explode()` directly on the series.  
Now we need to explode **and keep** the PC1, PC2, and taste_label columns aligned with each exploded row.

The trick is to explode the whole DataFrame (or a subset of it) and then work with the result:

```python
df_exploded = df_foodb[['food_sources', 'PC1', 'PC2', 'taste_label']].explode('food_sources')
```

After exploding, each row represents one (compound, food) pair, with the correct PC1/PC2/taste_label.

---

### ✏️ Exercise 10.1 — Create the exploded DataFrame

Fill in the steps below to prepare the data for plotting.

In [ ]:
# Step A: Select only the columns we need and drop rows with NaN taste or PCA coords
df_plot_base = df_foodb[['food_sources', 'PC1', 'PC2', 'taste_label']].dropna()

# YOUR CODE HERE
# Step B: Explode the 'food_sources' column so each row has a single food name
# Store the result in df_exploded


# YOUR CODE HERE  
# Step C: Drop rows where food_sources is NaN or an empty string after exploding
# Hint: use .dropna() and then filter out rows where food_sources == ''


print(f"Rows before exploding : {len(df_plot_base)}")
print(f"Rows after exploding  : {len(df_exploded)}")
print(f"(Each molecule appears once per food source it belongs to)")
print()
print("Preview:")
df_exploded.head(8)

<details>
<summary>💡 Hint</summary>

```python
df_exploded = df_plot_base.explode('food_sources')
df_exploded = df_exploded.dropna(subset=['food_sources'])
df_exploded = df_exploded[df_exploded['food_sources'] != '']
```
</details>

### ✏️ Exercise 10.2 — Plot the PCA grid by food source

Now create the same style of 3×3 scatter plot, but with one panel per **food source** instead of superclass.  
Use `top_foods` (computed in Step 4) as the list of food sources to plot.

**What you need to do:**
1. Loop over `top_foods` with index.
2. For each food, select rows from `df_exploded` where `food_sources == food`.
3. Plot bitter and sweet compounds in different colours, just like Step 9.
4. Set an informative title showing the food name and compound counts.

The figure setup and legend code is provided — fill in the loop.

In [ ]:
TASTE_COLORS = {'bitter': 'steelblue', 'sweet': 'coral'}

fig, axes = plt.subplots(3, 3, figsize=(15, 14))
axes = axes.flatten()

# YOUR CODE HERE
# Loop over top_foods (use enumerate to get both index and food name)
# For each food:
#   1. Get the current axis: ax = axes[idx]
#   2. Filter df_exploded to rows where food_sources == food
#      Store as 'subset'
#   3. For each taste ('bitter', 'sweet'), scatter plot the PC1/PC2 coords
#      using the matching colour from TASTE_COLORS
#      (alpha=0.45, s=12, linewidths=0)
#   4. Set a title like: f"{food}\n(n={n_total} | bitter={n_bitter} sweet={n_sweet})"
#   5. Set axis labels using var1 and var2 (already defined above)


# Shared legend
legend_handles = [
    mpatches.Patch(color=TASTE_COLORS['bitter'], label='Bitter (predicted)'),
    mpatches.Patch(color=TASTE_COLORS['sweet'],  label='Sweet (predicted)'),
]
fig.legend(
    handles=legend_handles,
    loc='lower center', ncol=2,
    fontsize=11, frameon=True,
    bbox_to_anchor=(0.5, -0.03)
)
fig.suptitle(
    "PCA of Morgan Fingerprints — coloured by predicted taste\n(grouped by food source)",
    fontsize=13, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()

<details>
<summary>💡 Hint</summary>

```python
for idx, food in enumerate(top_foods):
    ax = axes[idx]
    subset = df_exploded[df_exploded['food_sources'] == food]

    for taste, color in TASTE_COLORS.items():
        group = subset[subset['taste_label'] == taste]
        ax.scatter(
            group['PC1'], group['PC2'],
            c=color, alpha=0.45, s=12, linewidths=0
        )

    n_total  = len(subset)
    n_bitter = (subset['taste_label'] == 'bitter').sum()
    n_sweet  = (subset['taste_label'] == 'sweet').sum()

    ax.set_title(
        f"{food}\n(n={n_total} | bitter={n_bitter} sweet={n_sweet})",
        fontsize=9, fontweight='bold'
    )
    ax.set_xlabel(f"PC1 ({var1:.1f}% var)", fontsize=7)
    ax.set_ylabel(f"PC2 ({var2:.1f}% var)", fontsize=7)
    ax.tick_params(labelsize=6)
```
</details>

### ✏️ Exercise 10.3 — Compute and visualise bitter/sweet ratios per food

The scatter plots show *where* compounds sit in chemical space, but they don't make it easy to compare the **bitter-to-sweet ratio** across foods at a glance.

Create a **horizontal bar chart** where:
- Each bar represents one of the top 9 food sources.
- The bar is split into a **blue (bitter) segment** and an **orange/coral (sweet) segment**.
- Bars are sorted so the most bitter food is at the top.

This is a **stacked horizontal bar chart** (`ax.barh`).  
The table below summarises what you need to compute first:

| Column to compute | How |
|---|---|
| `n_bitter` per food | filter `taste_label == 'bitter'`, group by food, count |
| `n_sweet` per food | filter `taste_label == 'sweet'`, group by food, count |
| `pct_bitter` | `n_bitter / (n_bitter + n_sweet) * 100` |

In [ ]:
# YOUR CODE HERE
# Step A: For each food in top_foods, count bitter and sweet compounds
#
# Hint: filter df_exploded for top_foods, then group by 'food_sources' and 'taste_label'
#   df_top = df_exploded[df_exploded['food_sources'].isin(top_foods)]
#   taste_counts = df_top.groupby(['food_sources', 'taste_label']).size().unstack(fill_value=0)
#   This gives a DataFrame with foods as rows and 'bitter'/'sweet' as columns


# YOUR CODE HERE
# Step B: Compute pct_bitter (percentage of bitter compounds) and sort descending


# YOUR CODE HERE
# Step C: Plot a stacked horizontal bar chart
# - One row per food source
# - First bar segment: n_bitter (steelblue)
# - Second bar segment: n_sweet (coral), stacked after bitter
# - Add x-axis label 'Number of compounds'
# - Add a legend
# - Add a title
#
# Hint for stacked barh:
#   ax.barh(y_positions, bitter_values, color='steelblue', label='Bitter')
#   ax.barh(y_positions, sweet_values, left=bitter_values, color='coral', label='Sweet')


<details>
<summary>💡 Full solution</summary>

```python
# Step A
df_top = df_exploded[df_exploded['food_sources'].isin(top_foods)]
taste_counts = df_top.groupby(['food_sources', 'taste_label']).size().unstack(fill_value=0)

# Ensure both columns exist
for col in ['bitter', 'sweet']:
    if col not in taste_counts.columns:
        taste_counts[col] = 0

# Step B
taste_counts['pct_bitter'] = taste_counts['bitter'] / (taste_counts['bitter'] + taste_counts['sweet']) * 100
taste_counts = taste_counts.sort_values('pct_bitter', ascending=True)  # ascending=True so most bitter is at top

# Step C
fig, ax = plt.subplots(figsize=(10, 5))
y = range(len(taste_counts))
ax.barh(y, taste_counts['bitter'], color='steelblue', label='Bitter')
ax.barh(y, taste_counts['sweet'], left=taste_counts['bitter'], color='coral', label='Sweet')
ax.set_yticks(y)
ax.set_yticklabels(taste_counts.index)
ax.set_xlabel('Number of compounds')
ax.set_title('Bitter vs Sweet compound counts by food source (Top 9)')
ax.legend()
plt.tight_layout()
plt.show()
```
</details>

---
## Step 10 — Reflection Questions

After completing the food-source PCA, take a moment to think about these questions:

1. **Bitter vs sweet by food:** Which of the top 9 foods has the highest proportion of predicted-bitter compounds?
   Does that match your intuition about that food?

2. **Comparing superclass vs food source plots:** The superclass plot (Step 9) shows what we would expect, while the food-source plot is difficult to interpret.
   Why might that be?

3. **Limitations:** The model was trained on a specific dataset. What kinds of errors might it make on FooDB compounds?
   Can you think of a way to check?

---
## Summary — What You Built

Congratulations! Here is the full pipeline you just implemented:

1. **Loaded** the FooDB compound database.
2. **Parsed** the food source lists stored as text strings back into real Python lists.
3. **Counted** how many compounds come from each food source and plotted a bar chart.
4. **Loaded** a pre-trained bitter/sweet Random Forest classifier from disk using `pickle`.
5. **Converted** every SMILES string into a Morgan fingerprint (1024-bit binary vector) using RDKit.
6. **Predicted** whether each compound tastes bitter or sweet using `model.predict()`.
7. **Compressed** the 1024-dimensional fingerprints to 2D using PCA.
8. **Plotted** PCA panels coloured by predicted taste, both by **compound superclass** and by **food source**.
9. **Compared** bitter/sweet ratios across food sources with a stacked bar chart.